In [1]:
!git clone https://github.com/aliakseizvertouski/olist.git

Cloning into 'olist'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 67 (delta 33), reused 6 (delta 3), pack-reused 14 (from 1)
Receiving objects: 100% (67/67), 42.62 MiB | 17.24 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
customers = pd.read_csv('/content/olist/olist_customers_dataset.csv')
geo = pd.read_csv('/content/olist/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/olist/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/olist/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/olist/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/olist/olist_orders_dataset.csv')
products = pd.read_csv('/content/olist/olist_products_dataset.csv')
sellers = pd.read_csv('/content/olist/olist_sellers_dataset.csv')

# Информация о датасете

In [4]:
print (customers.columns)
print (geo.columns)
print (order_items.columns)
print (order_payments.columns)
print (order_reviews.columns)
print (orders.columns)
print (products.columns)
print (sellers.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['product_id', 'prod

# Анализ продаж


In [42]:
df_prices = order_items[['order_id', 'order_item_id', 'product_id', 'seller_id', 'price']].merge(
    orders[['order_id', 'order_purchase_timestamp']],
    on='order_id'
)

df_prices['purchase_date'] = pd.to_datetime(df_prices['order_purchase_timestamp'])
df_prices = df_prices.sort_values(by=['seller_id', 'product_id', 'purchase_date'])


df_prices['prev_price'] = df_prices.groupby(['seller_id', 'product_id'])['price'].shift(1)
df_prices['discount_percent'] = ((df_prices['prev_price'] - df_prices['price']) / df_prices['prev_price']) * 100
df_prices['discount_percent'] = df_prices['discount_percent'].fillna(0)

df_prices = df_prices.drop(columns=['order_id'])

def f(arr):
    rank = 0
    values = arr.tolist()
    result = [0] * len(values)
    for idx, value in enumerate(values):
        if value != values[idx - 1]:
            rank += 1
        result[idx] = rank
    return pd.Series(result, index=arr.index)

df_prices['price_period_rank'] = df_prices.groupby(['seller_id', 'product_id'])['price'].transform(f)

df_prices['sales'] = df_prices.groupby(['seller_id', 'product_id', 'price_period_rank'])['price_period_rank'].transform('size')

# df_prices = df_prices[df_prices['price_period_size'] >= 3]

df_prices = df_prices[df_prices['discount_percent'] != 0]

# df_prices = df_prices.groupby(['product_id', 'seller_id'])['sales'].shift(1)

df_prices['next_sales'] = df_prices.groupby(['seller_id', 'product_id'])['sales'].shift(-1)

df_prices


,order_item_id,product_id,seller_id,price,order_purchase_timestamp,purchase_date,prev_price,discount_percent,price_period_rank,sales,next_sales
85473,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,89.00,2017-04-10 10:41:42,2017-04-10 10:41:42,69.90,-27.324750,2,2,1.0
75646,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,135.00,2017-05-02 15:23:09,2017-05-02 15:23:09,89.00,-51.685393,3,1,13.0
78595,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,99.00,2017-05-05 22:12:04,2017-05-05 22:12:04,135.00,26.666667,4,13,25.0
25695,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,89.00,2017-05-23 11:58:50,2017-05-23 11:58:50,99.00,10.101010,5,25,60.0
70829,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,99.00,2017-08-14 10:07:18,2017-08-14 10:07:18,89.00,-11.235955,6,60,5.0
...,...,...,...,...,...,...,...,...,...,...,...
11109,1,7766894470ea995b418764065e6bf9ba,ffc470761de7d0232558ba5e786e57b7,26.68,2018-07-19 11:38:46,2018-07-19 11:38:46,24.98,-6.805444,2,1,NaN
45,1,dbaee28f4ee64465838a229582d77520,ffc470761de7d0232558ba5e786e57b7,54.00,2018-07-04 11:39:11,2018-07-04 11:39:11,54.90,1.639344,2,4,NaN
62708,1,58a96ae04b90563cde5becdf0ee2e823,ffdd9f82b9a447f6f8d4b91554cc7dd3,199.00,2017-10-26 14:49:40,2017-10-26 14:49:40,206.00,3.398058,2,1,NaN
103154,1,ada800a927673ac73cdfbbd2c832331b,ffdd9f82b9a447f6f8d4b91554cc7dd3,63.60,2018-04-17 19:19:04,2018-04-17 19:19:04,53.60,-18.656716,2,2,1.0


In [39]:
target_seller = '001cca7ae9ae17fb1caed9dfb1094831'
target_product = '08574b074924071f4e201e151b152b4e'

target_seller_data = df_prices[df_prices['seller_id'] == target_seller]
target_product_data = target_seller_data[target_seller_data['product_id'] == target_product]


